In [ ]:
import pandas as pd
import numpy as np
import gzip
from io import StringIO
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, MultiHeadAttention, LayerNormalization, add, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.metrics import Precision, Recall

In [ ]:
# 读取并预处理数据
def read_data(filename):
    threshold = 1800
    max_tokens = 27

    def pad_sequence(sequence):
        return sequence + "-" * (threshold - len(sequence))

    file_path = filename + '.tsv.gz'
    with gzip.open(file_path, 'rt') as f:
        content = f.read()

    df = pd.read_csv(StringIO(content), sep='\t')
    df = df.dropna(subset=['Gene Ontology (GO)', 'Sequence'])
    df = df[~df["Sequence"].duplicated()]
    df = df[df['Sequence'].apply(lambda x: len(x)) <= threshold]
    df.reset_index(drop=True, inplace=True)
    df['Sequence'] = df['Sequence'].apply(pad_sequence)

    char_vals = {"M":1,"N":2,"S":3,"K":4,"I":5,"F":6,"A":7,"V":8,"L":9,"G":10,
                 "R":11,"C":12,"D":13,"Q":14,"Y":15,"P":16,"T":17,"E":18,
                 "H":19,"W":20,"X":21,"U":22,"Z":23,"B":24,"O":25,"-":26}
    df['Sequence'] = df['Sequence'].apply(lambda x: [char_vals[char] for char in list(x)])

    return df

In [ ]:
# 构建Transformer模型
def build_transformer_model(input_shape, max_tokens, embed_dim, num_heads):
    inputs = Input(shape=input_shape)
    embed = Embedding(input_dim=max_tokens, output_dim=embed_dim)(inputs)

    x = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(embed, embed)
    x = add([x, embed])
    x = LayerNormalization()(x)

    x = Dense(embed_dim, activation="relu")(x)
    x = add([x, embed])
    x = GlobalAveragePooling1D()(x)

    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)

    return model

In [ ]:
# 构建LSTM模型
def build_lstm_model(input_shape, max_tokens, embed_dim, lstm_units):
    inputs = Input(shape=input_shape)
    embed = Embedding(input_dim=max_tokens, output_dim=embed_dim)(inputs)

    x = LSTM(units=lstm_units)(embed)

    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)

    return model

In [ ]:
# 链接Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 读取并预处理数据
dev_df_positive = read_data("drive/MyDrive/Extracellular_matrix_organization")
dev_df_negative = read_data("drive/MyDrive/Not_extracellular_matrix_organization")

dev_df_positive['target'] = 1
dev_df_negative['target'] = 0

dev_df_positive = dev_df_positive.iloc[:40000]
dev_df_negative = dev_df_negative.iloc[:40000]

dev_df = pd.concat([dev_df_positive, dev_df_negative])
dev_df = dev_df.sample(frac=1).reset_index(drop=True)

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(dev_df['Sequence'], dev_df['target'], test_size=0.2)

X_train = np.array(X_train.tolist())
X_val = np.array(X_val.tolist())

# 构建并训练Transformer模型
transformer_model = build_transformer_model(input_shape=(1800,), max_tokens=27, embed_dim=100, num_heads=4)
transformer_model.compile(optimizer=RMSprop(learning_rate=0.01), loss="binary_crossentropy", metrics=["acc", Precision(), Recall()])
transformer_model.fit(X_train, y_train, epochs=1, batch_size=64, validation_data=(X_val, y_val))

# 构建并训练LSTM模型
lstm_model = build_lstm_model(input_shape=(1800,), max_tokens=27, embed_dim=100, lstm_units=64)
lstm_model.compile(optimizer=RMSprop(learning_rate=0.01), loss="binary_crossentropy", metrics=["acc", Precision(), Recall()])
lstm_model.fit(X_train, y_train, epochs=1, batch_size=64, validation_data=(X_val, y_val))

# 保存模型
transformer_model.save('transformer_ecm_model.keras')
lstm_model.save('lstm_ecm_model.keras')

1000/1000 [==============================] - 68s 66ms/step - loss: 0.6936 - acc: 0.5011 - precision_2: 0.5003 - recall_2: 0.4390 - val_loss: 0.6936 - val_acc: 0.5032 - val_precision_2: 0.5032 - val_recall_2: 1.0000
